In [3]:
# ===================== INSTALL =====================
!pip install -q gspread pandas numpy

# ===================== IMPORTS =====================
import pandas as pd
import numpy as np
import gspread
from google.colab import auth
from google.auth import default

# ===================== AUTH =====================
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# ===================== LOAD LAST 5 ROWS =====================
SHEET_ID = "1aT0qHpKQUcV7jelT12prvHdHzwuNl19_Wf_-N9F2-30"
sheet = gc.open_by_key(SHEET_ID).sheet1

rows = sheet.get_all_values()
headers = rows[0]  # First row = headers
data = rows[1:]    # Rest = data

df = pd.DataFrame(data, columns=headers)

# Take last 5 rows
df = df.tail(5)

# ===================== PROCESS RED =====================
# Convert Red column to numeric
df['R'] = pd.to_numeric(df['R'], errors='coerce')

# Drop rows with invalid Red values
df = df.dropna(subset=['R'])

# Normalize Red
def normalize(x):
    x = np.array(x, dtype=float)
    if x.size == 0:
        return np.array([])
    min_val = np.min(x)
    max_val = np.max(x)
    if max_val == min_val:
        return np.zeros_like(x)
    return (x - min_val) / (max_val - min_val)

df['R_norm'] = normalize(df['R'].values)

# ===================== CALCULATE GLUCOSE =====================
# Simple formula based on Red only
df['Predicted_Glucose'] = 90 + 50 * df['R_norm']  # baseline + factor * normalized Red

# Glucose status
def glucose_status(g):
    if g < 70:
        return "LOW"
    elif g <= 140:
        return "NORMAL"
    elif g <= 180:
        return "HIGH"
    else:
        return "VERY HIGH"

df['Status'] = df['Predicted_Glucose'].apply(glucose_status)

# ===================== FINAL OUTPUT =====================
final_output = df[['R', 'R_norm', 'Predicted_Glucose']]
final_output

,R,R_norm,Predicted_Glucose
176,13064.10,1.000000,140.000000
177,12839.48,0.361258,108.062902
178,12861.57,0.424074,111.203720
179,12784.25,0.204203,100.210146
180,12712.44,0.000000,90.000000
